In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, TimeSeriesSplit
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

In [22]:
path = r'..\Data\Original_Data\Motot_Type_YWNC-203.xlsx'
df = pd.read_excel(path)
df.head()

,Time,TOTAL_NET_KWH,AVG_CURRENT,AVG_V_LL,AVG_V_LN,FREQUENCY,Type
0,"22/07/2025, 20:55:44",15422.430,2.445,411.948,237.957,50.024,YWNC-203
1,"22/07/2025, 21:00:48",15422.520,2.445,411.544,237.618,50.005,YWNC-203
2,"22/07/2025, 21:06:00",15422.613,2.428,411.182,237.445,50.008,YWNC-203
3,"22/07/2025, 21:11:06",15422.703,2.445,412.419,238.222,49.982,YWNC-203
4,"22/07/2025, 21:20:52",15422.779,0.445,411.297,237.482,50.031,YWNC-203


In [23]:
df.shape

(47879, 7)

In [24]:
df[df.duplicated]

,Time,TOTAL_NET_KWH,AVG_CURRENT,AVG_V_LL,AVG_V_LN,FREQUENCY,Type
36633,"07/12/2025, 23:55:33",11309.544,0.62,416.007,240.198,49.988,YWNC2 CONE


In [25]:
df.drop_duplicates(inplace=True)

In [26]:
df.isnull().sum()

Time             0
TOTAL_NET_KWH    0
AVG_CURRENT      0
AVG_V_LL         0
AVG_V_LN         0
FREQUENCY        0
Type             0
dtype: int64

In [27]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
TOTAL_NET_KWH,47878.0,11303.883482,2308.859143,8388.642,9705.83950,10816.0365,11653.38725,16777.183
AVG_CURRENT,47878.0,3.992462,1.999624,0.406,2.70800,4.0690,4.94700,13.648
AVG_V_LL,47878.0,402.217167,5.830307,384.230,398.52725,400.2080,402.90975,430.847
AVG_V_LN,47878.0,232.256325,3.357195,221.906,230.13800,231.1000,232.64900,248.755
FREQUENCY,47878.0,49.996883,0.064474,49.424,49.96400,49.9990,50.03400,50.396


In [28]:
mask = df["Time"].astype(str).str.contains(", 24:")

# fix time: replace 24 with 00
df.loc[mask, "Time"] = (
    df.loc[mask, "Time"]
      .str.replace(", 24:", ", 00:", n=1)
)

# Convert datetime type data
df["Time"] = pd.to_datetime(df["Time"], format="%d/%m/%Y, %H:%M:%S")

# Short by time (Important for energy data)
df = df.sort_values('Time')

# Set time as index
df = df.set_index('Time')

In [29]:
df['KWH_diff'] = df["TOTAL_NET_KWH"].diff()

In [30]:
## Replacing the -ve KWH_diff with nan

df['KWH_diff'] = df['KWH_diff'].where(df['KWH_diff'] >= 0)

# Drop the nan value
df = df.dropna(subset=['KWH_diff'])

In [31]:
df.isnull().sum()

TOTAL_NET_KWH    0
AVG_CURRENT      0
AVG_V_LL         0
AVG_V_LN         0
FREQUENCY        0
Type             0
KWH_diff         0
dtype: int64

In [32]:
df_energy = df[['KWH_diff']].copy()

In [33]:
def split_large_gaps(df, threshold=1.0):
    df = df.sort_index()

    rows_to_add = []

    # recompute gap hours EVERY iteration
    gap_hours = df.index.to_series().diff().dt.total_seconds() / 3600

    for i in range(1, len(df)):
        gap = gap_hours.iloc[i]

        if gap > threshold:
            prev_time = df.index[i - 1]
            curr_time = df.index[i]

            midpoint = prev_time + (curr_time - prev_time) / 2

            energy = df.iloc[i]['KWH_diff'] / 2

            # halve current row energy
            df.at[curr_time, 'KWH_diff'] = energy

            # create midpoint row
            new_row = df.loc[curr_time].copy()
            new_row.name = midpoint
            new_row['KWH_diff'] = energy

            # invalidate non-energy features
            for col in ['AVG_CURRENT', 'AVG_V_LL', 'AVG_V_LN', 'FREQUENCY']:
                if col in new_row:
                    new_row[col] = np.nan

            rows_to_add.append(new_row)

    if not rows_to_add:
        return df, False

    df = pd.concat([df, pd.DataFrame(rows_to_add)]).sort_index()

    return df, True

In [34]:
while True:
    df, changed = split_large_gaps(df, threshold=1.0)
    if not changed:
        break

In [35]:
time_diff_index = df.index.to_series().diff()

df[time_diff_index > pd.Timedelta(minutes=60)]

,TOTAL_NET_KWH,AVG_CURRENT,AVG_V_LL,AVG_V_LN,FREQUENCY,Type,KWH_diff


In [36]:
hourly_df = (
    df
    .resample('1H')
    .agg({
        'KWH_diff': 'sum',
        'AVG_CURRENT': 'mean',
        'AVG_V_LN': 'mean'
    })
)

C:\Users\devan\AppData\Local\Temp\ipykernel_17852\786433140.py:3: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample('1H')


In [37]:
hourly_df.rename(columns={'KWH_diff': 'HOURLY_KWH'}, inplace=True)

In [38]:
hourly_df['HOURLY_KWH'] = hourly_df['HOURLY_KWH'].replace(0.0, np.nan)

In [39]:
hourly_df.isnull().sum()

HOURLY_KWH       0
AVG_CURRENT    176
AVG_V_LN       176
dtype: int64

In [40]:
hourly_df[hourly_df["HOURLY_KWH"] == 0.0]

,HOURLY_KWH,AVG_CURRENT,AVG_V_LN


In [41]:
# Adding a gap flag

hourly_df['long_gap_flag'] = hourly_df['AVG_CURRENT'].isna() & hourly_df['AVG_V_LN'].isna()

In [42]:
cols = ['AVG_CURRENT', 'AVG_V_LN']

for col in cols:
    hourly_df[col] = hourly_df[col].fillna(hourly_df[col].median())

In [43]:


hourly_df['hour'] = hourly_df.index.hour
hourly_df['day'] = hourly_df.index.day
hourly_df['weekday'] = hourly_df.index.weekday

## EDA

In [44]:
print(f"Shape of the dataset: {hourly_df.shape}")
print("_"*40)
print("Information:- ")
print(hourly_df.info())

Shape of the dataset: (3828, 7)
________________________________________
Information:- 
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3828 entries, 2025-07-22 21:00:00 to 2025-12-29 08:00:00
Freq: h
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   HOURLY_KWH     3828 non-null   float64
 1   AVG_CURRENT    3828 non-null   float64
 2   AVG_V_LN       3828 non-null   float64
 3   long_gap_flag  3828 non-null   bool   
 4   hour           3828 non-null   int32  
 5   day            3828 non-null   int32  
 6   weekday        3828 non-null   int32  
dtypes: bool(1), float64(3), int32(3)
memory usage: 168.2 KB
None


In [45]:
hourly_df.head()

,HOURLY_KWH,AVG_CURRENT,AVG_V_LN,long_gap_flag,hour,day,weekday
2025-07-22 21:00:00,0.507,1.151818,234.739273,False,21,22,1
2025-07-22 22:00:00,1.197,2.508750,233.409417,False,22,22,1
2025-07-22 23:00:00,1.242,2.655833,234.676667,False,23,22,1
2025-07-23 00:00:00,1.901,3.879091,234.648636,False,0,23,2
2025-07-23 01:00:00,1.187,2.545000,235.833000,False,1,23,2


In [46]:
hourly_df.tail()

,HOURLY_KWH,AVG_CURRENT,AVG_V_LN,long_gap_flag,hour,day,weekday
2025-12-29 04:00:00,0.154,0.555087,239.906783,False,4,29,0
2025-12-29 05:00:00,0.156,0.554348,239.902261,False,5,29,0
2025-12-29 06:00:00,0.148,0.560955,238.226591,False,6,29,0
2025-12-29 07:00:00,1.294,4.616652,232.996217,False,7,29,0
2025-12-29 08:00:00,0.640,4.465818,231.181091,False,8,29,0


In [47]:
hourly_df.isnull().sum()

HOURLY_KWH       0
AVG_CURRENT      0
AVG_V_LN         0
long_gap_flag    0
hour             0
day              0
weekday          0
dtype: int64

In [48]:
hourly_df.describe().T

,count,mean,std,min,25%,50%,75%,max
HOURLY_KWH,3828.0,1.263437,0.838903,0.000563,0.947000,1.293000,1.475000,29.936563
AVG_CURRENT,3828.0,4.024868,1.663864,0.415273,3.497167,4.610091,5.099563,7.826909
AVG_V_LN,3828.0,232.183677,3.297570,224.383091,230.229513,230.982083,232.200758,246.056833
hour,3828.0,11.490596,6.929196,0.000000,5.000000,11.000000,17.250000,23.000000
day,3828.0,16.204545,8.878662,1.000000,8.000000,16.000000,24.000000,31.000000
weekday,3828.0,3.022727,1.995227,0.000000,1.000000,3.000000,5.000000,6.000000


In [49]:
hourly_df[hourly_df['HOURLY_KWH'] > 4]

,HOURLY_KWH,AVG_CURRENT,AVG_V_LN,long_gap_flag,hour,day,weekday
2025-12-09 14:00:00,29.936563,4.423727,231.933818,False,14,9,1


In [50]:
hourly_df.loc['2025-12-09 12:00:00':'2025-12-09 15:00:00']

,HOURLY_KWH,AVG_CURRENT,AVG_V_LN,long_gap_flag,hour,day,weekday
2025-12-09 12:00:00,0.001125,4.610091,230.982083,True,12,9,1
2025-12-09 13:00:00,0.000563,4.610091,230.982083,True,13,9,1
2025-12-09 14:00:00,29.936563,4.423727,231.933818,False,14,9,1
2025-12-09 15:00:00,1.694000,5.877391,230.661130,False,15,9,1


In [51]:
## Histogram -- Distribution of Hourly Energy Consumption

fig = px.histogram(
    hourly_df['HOURLY_KWH'],
    nbins=50,
    labels={'value': 'HOURLY_KWH'},
    title='Distribution of Hourly Energy Consumption',
    opacity=0.85
)

fig.update_layout(
    bargap=0.05,
    xaxis_title='HOURLY_KWH',
    yaxis_title='Count',
    template='plotly_dark'
)

fig.show()

In [52]:
## Boxplot of Hourly_KWH

fig = px.box(
    hourly_df,
    y='HOURLY_KWH',
    points='outliers', 
    title='Boxplot of HOURLY_KWH'
)

fig.update_layout(
    yaxis_title='HOURLY_KWH',
    template='plotly_dark'
)

fig.show()

In [53]:
fig = px.line(
    hourly_df,
    x=hourly_df.index,
    y='HOURLY_KWH',
    title='Hourly Energy Consumption Over Time'
)

fig.update_layout(
    xaxis_title='Time',
    yaxis_title='HOURLY_KWH',
    template='plotly_dark',
    hovermode='x unified'
)

fig.show()

In [54]:
## Average HOURLY_KWH

hourly_avg = hourly_df.groupby('hour', as_index=False)['HOURLY_KWH'].mean()

fig = px.line(
    hourly_avg,
    x='hour',
    y='HOURLY_KWH',
    markers=True,
    title='Average HOURLY_KWH by Hour'
)

fig.update_layout(
    xaxis_title='Hour of Day',
    yaxis_title='Average HOURLY_KWH',
    template='plotly_dark',
    hovermode='x unified'
)

fig.update_xaxes(
    tickmode='linear',
    dtick=1
)

fig.show()

In [55]:
## Average Energy Consumption by Weekday

weekday_avg = (
    hourly_df
    .groupby('weekday', as_index=False)['HOURLY_KWH']
    .mean()
)

fig = px.bar(
    weekday_avg,
    x='weekday',
    y='HOURLY_KWH',
    title='Average Energy Consumption by Weekday',
    text_auto='.2f'
)

fig.update_layout(
    xaxis_title='Weekday (0=Mon, 6=Sun)',
    yaxis_title='Average HOURLY_KWH',
    template='plotly_dark',
    hovermode='x unified'
)

fig.update_xaxes(
    tickmode='linear',
    dtick=1
)

fig.show()

In [56]:
## Average Daily Energy Consumption Pattern

daily_avg = (
    hourly_df
    .groupby('day', as_index=False)['HOURLY_KWH']
    .mean()
)

fig = px.line(
    daily_avg,
    x='day',
    y='HOURLY_KWH',
    markers=True,
    title='Average Daily Energy Consumption Pattern'
)

fig.update_layout(
    xaxis_title='Day of Month',
    yaxis_title='Average HOURLY_KWH',
    template='plotly_dark',
    hovermode='x unified'
)

fig.update_xaxes(
    tickmode='linear',
    dtick=1
)

fig.show()

In [ ]:
# Scatterplot - HOURLY_KWH vs Rest of the columns

cols = ['AVG_CURRENT','AVG_V_LN', 'long_gap_flag']

for col in cols:
    fig = px.scatter(
        hourly_df,
        x=col,
        y='HOURLY_KWH',
        opacity=0.5,
        title=f'HOURLY_KWH vs {col}',
        hover_data=['hour', 'day', 'weekday']
    )

    fig.update_layout(
        xaxis_title=col,
        yaxis_title='HOURLY_KWH',
        template='plotly_dark'
    )

    fig.show()

## Feature Engineering

In [58]:
# Convert hour & weekday to sine–cosine
##This helps ML models understand continuity (23 → 0).

hourly_df["hour_sin"] = np.sin(2 * np.pi * hourly_df["hour"] / 24)
hourly_df["hour_cos"] = np.cos(2 * np.pi * hourly_df["hour"] / 24)

hourly_df["weekday_sin"] = np.sin(2 * np.pi * hourly_df["weekday"] / 7)
hourly_df["weekday_cos"] = np.cos(2 * np.pi * hourly_df["weekday"] / 7)

In [59]:
# Lag Features (MOST IMPORTANT for Time Series)
## Energy depends heavily on past consumption.

for lag in [1, 8, 24, 168]:
    hourly_df[f"kwh_lag_{lag}"] = hourly_df["HOURLY_KWH"].shift(lag)

In [60]:
hourly_df["is_7to15_shift"] = (
    (hourly_df["hour"] >= 7) & (hourly_df["hour"] < 15)
).astype(int)

In [61]:
# Rolling Statistics (Smooths Noise)
##From your plots, energy has local fluctuations

hourly_df["kwh_roll_2"] = hourly_df["HOURLY_KWH"].rolling(2).mean()
hourly_df["kwh_roll_24"] = hourly_df["HOURLY_KWH"].rolling(24).mean()

In [62]:
# Interaction Features (Energy ≠ single variable)

hourly_df["power_proxy"] = hourly_df["AVG_CURRENT"] * hourly_df["AVG_V_LN"]

In [63]:
# Time since last valid reading

gap_id = hourly_df["long_gap_flag"].cumsum()

hourly_df["time_since_gap"] = (
    hourly_df
    .groupby(gap_id)
    .cumcount()
)

hourly_df["time_since_gap_log"] = np.log1p(hourly_df["time_since_gap"])

In [64]:
# Is weekend

hourly_df["is_sunday"] = (hourly_df["weekday"] == 6).astype(int)

In [65]:
# Low-usage

rolling_baseline = hourly_df["HOURLY_KWH"].rolling(
    window=24,
    min_periods=12
).mean()

hourly_df["low_activity_detected"] = (
    hourly_df["HOURLY_KWH"] < 0.25 * rolling_baseline
).astype(int)




In [66]:
#  Spike Detection

rolling_median = hourly_df["HOURLY_KWH"].rolling(
    window=24,
    min_periods=12
).median()

mad = (
    (hourly_df["HOURLY_KWH"] - rolling_median)
    .abs()
    .rolling(24, min_periods=12)
    .median()
)

hourly_df["spike_detected"] = (
    hourly_df["HOURLY_KWH"] > (rolling_median + 3 * mad)
).astype(int)

In [67]:
## Correlation Heatmap

corr_matrix = hourly_df.corr()

fig = px.imshow(
    corr_matrix,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu",
    title="Correlation Heatmap"
)

fig.update_layout(
    template="plotly_dark",
    xaxis_title="Features",
    yaxis_title="Features"
)

fig.show()

In [76]:
## Calculating Outliers

Q1 = hourly_df['HOURLY_KWH'].quantile(0.25)
Q3 = hourly_df['HOURLY_KWH'].quantile(0.75)
IQR = Q3 - Q1
lower_limit = Q1 - 1.5*IQR
upper_limit = Q3 + 1.5*IQR

outliers = hourly_df[
    (hourly_df['HOURLY_KWH'] < lower_limit) |
    (hourly_df['HOURLY_KWH'] > upper_limit)
]
outliers.shape

(709, 24)

In [77]:
fig = go.Figure()

# Raw HOURLY_KWH
fig.add_trace(go.Scatter(
    x=hourly_df.index,
    y=hourly_df['HOURLY_KWH'],
    mode='lines',
    name='HOURLY_KWH',
    line=dict(color='lightblue'),
    opacity=0.3
))

# 24h Rolling Mean
fig.add_trace(go.Scatter(
    x=hourly_df.index,
    y=hourly_df['kwh_roll_24'],
    mode='lines',
    name='24h Rolling Mean',
    line=dict(color='orange', width=2)
))

fig.update_layout(
    title='Hourly Energy Consumption with 24h Rolling Mean',
    xaxis_title='Time',
    yaxis_title='HOURLY_KWH',
    template='plotly_dark',
    hovermode='x unified'
)

fig.show()

In [78]:
len(hourly_df)

3828

#### Handling Outliers
Cap the data to the upper limit and the lower limit

In [80]:
hourly_df['HOURLY_KWH_capped'] = hourly_df['HOURLY_KWH'].clip(lower=lower_limit, upper=upper_limit)

In [81]:
## Before and after capping
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=hourly_df.index, y=hourly_df['HOURLY_KWH'], mode='lines', name='Original', opacity=0.3))
fig.add_trace(go.Scatter(
    x=hourly_df.index, y=hourly_df['HOURLY_KWH_capped'], mode='lines', name='Capped', line=dict(color='orange')))
fig.show()

In [83]:
hourly_df.to_csv(r"..\Data\HourlyTransformedData\hourly_energy_data_for_YWNC-203.csv")
hourly_df.to_pickle(r"..\Data\HourlyTransformedData\hourly_energy_data_for_YWNC-203.pkl")